# Demo: Automatic Evaluation and Error Detection in English-to-French translation using Nural Models

Dataset: IIT Bombay English-Hindi Corpus https://www.cfilt.iitb.ac.in/iitb_parallel/

In [2]:
!pip install openai datasets nltk transformers sentencepiece

In [3]:
import os
from transformers import pipeline #HuggingFace pipeline
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

In [4]:
import pandas as pd
from datasets import load_dataset
from openai import OpenAI
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
#Load Dataset
dataset = load_dataset("cfilt/iitb-english-hindi")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [6]:
#Take sample for demo
df = pd.DataFrame(dataset['test'][:20])
print(df)

                                          translation
0   {'en': 'A black box in your car?', 'hi': 'आपकी...
1   {'en': 'As America's road planners struggle to...
2   {'en': 'The devices, which track every mile a ...
3   {'en': 'The usually dull arena of highway plan...
4   {'en': 'Libertarians have joined environmental...
5   {'en': 'The tea party is aghast.', 'hi': 'चाय ...
6   {'en': 'The American Civil Liberties Union is ...
7   {'en': 'And while Congress can't agree on whet...
8   {'en': 'They are exploring how, over the next ...
9   {'en': 'Thousands of motorists have already ta...
10  {'en': 'This really is a must for our nation.'...
11  {'en': '"It is not a matter of something we mi...
12  {'en': 'There is going to be a change in how w...
13  {'en': 'The technology is there to do it.', 'h...
14  {'en': 'The push comes as the country's Highwa...
15  {'en': 'Americans don't buy as much gas as the...
16  {'en': 'Cars get many more miles to the gallon...
17  {'en': 'The federal tax 

In [7]:
df["English"]=df["translation"].apply(lambda x:x['en'])
df["Hindi_ref"]=df["translation"].apply(lambda x:x['hi'])
df=df[['English','Hindi_ref']]
print(df)

                                              English  \
0                            A black box in your car?   
1   As America's road planners struggle to find th...   
2   The devices, which track every mile a motorist...   
3   The usually dull arena of highway planning has...   
4   Libertarians have joined environmental groups ...   
5                            The tea party is aghast.   
6   The American Civil Liberties Union is deeply c...   
7   And while Congress can't agree on whether to p...   
8   They are exploring how, over the next decade, ...   
9   Thousands of motorists have already taken the ...   
10              This really is a must for our nation.   
11  "It is not a matter of something we might choo...   
12  There is going to be a change in how we pay th...   
13                  The technology is there to do it.   
14  The push comes as the country's Highway Trust ...   
15   Americans don't buy as much gas as they used to.   
16            Cars get many mor

In [8]:
client=OpenAI()
def translate_opeai(text):
    response=client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
                {"role": "system","content":"Please Translate English to Hindi."},
                {"role": "user","content":text}
            ]
        )
    return response.choices[0].message.content.strip()

df['Hindi_openai']=df['English'].apply(translate_opeai)
print(df['Hindi_openai'])

0                          आपकी कार में एक ब्लैक बॉक्स?
1     जब अमेरिका के सड़क योजनाकार एक टूटते हाईवे सिस...
2     ये उपकरण, जो हर मील को ट्रैक करते हैं जिसे एक ...
3     सड़क योजना का आमतौर पर नीरस क्षेत्र अचानक तीव्...
4     लिबर्टेरियन ने पर्यावरण समूहों के साथ मिलकर लॉ...
5                               चाय की पार्टी हैरान है।
6     अमेरिकी नागरिक स्वतंत्रता संघ भी गहरी चिंता मे...
7     और जबकि कांग्रेस आगे बढ़ने पर सहमत नहीं है, कई...
8     वे यह जांच रहे हैं कि कैसे, अगले दशक में, वे ए...
9     हजारों मोटर चालकों ने पहले ही काले बॉक्स ले लि...
10       यह वास्तव में हमारे देश के लिए एक आवश्यकता है।
11    “यह कुछ ऐसा नहीं है जिसे हम करने के लिए चुन सक...
12    इन करों को चुकाने के तरीके में बदलाव होने वाला...
13                 यहाँ इसे करने के लिए तकनीक मौजूद है।
14    यह दबाव तब आया है जब देश का हाईवे ट्रस्ट फंड, ...
15    अमेरिकियों ने पहले की तरह गैस खरीदना कम कर दिय...
16          गाड़ियों को गैलन में और अधिक मील मिलते हैं।
17    संघीय कर स्वयं, प्रति गैलन 18.4 सेंट, पिछल

### Compare with MarianMT-English--->Hindi Model


In [12]:
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util

model_name = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate_marianmt(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

df["Hindi_marianmt"]=df["English"].apply(translate_marianmt)

print(df["Hindi_marianmt"])

0                          अपनी कार में एक ब्लैक बॉक्स?
1     के रूप में अमेरिका के सड़क योजनार एक टूटी हुई ...
2     ये उपकरण, जो हर मील का ट्रैक करते हैं एक मोटर ...
3     आम तौर पर सड़क योजना के ठंडे रंग में बहस और रं...
4     कुछ देशों में ऐसे भी लोग हैं जो एक - दूसरे को ...
5                               चाय पार्टी कम हो गई है.
6     अमरीकी नागरिक संघ को भी अलग - अलग निजी मामलों ...
7     और जबकि कांग्रेस आगे बढ़ने पर सहमत नहीं हो सकत...
8     वे कल्पना कर रहे हैं कि अगले दस सालों के दौरान...
9     हज़ारों मोटर - चालकों ने पहले ही ब्लैक बक्से ल...
10       यह वास्तव में हमारी जाति के लिए एक होना चाहिए.
11    "यह कोई बात नहीं है जो हम करने के लिए चुन सकते...
12    हम इन कर का भुगतान कैसे कर रहे हैं में एक परिव...
13                      प्रौद्योगिकी यह करने के लिए है.
14    धक्का देश के उच्चवे पर भरोसा निधि के रूप में आ...
15    अमरीकी उतना गैस नहीं खरीदते जितना वे इस्तेमाल ...
16        झूठ बोलने के लिए कार को और अधिक मील मिलता है.
17    संघीय कर अपने आप, 184 सेंट प्रति सेंट, 20 

In [10]:
references=[[ref.split()] for ref in df['Hindi_ref']]

#BLEU score OpenAI
cand_openai=[pred.split() for pred in df['Hindi_openai']]
bleu_openai=corpus_bleu(references,cand_openai)
print(f"BLEU Score OpenAI: {bleu_openai}")

BLEU Score OpenAI: 0.12679178772067543


In [11]:
#BLEU score for MarianMT
cand_marianmt=[pred.split() for pred in df['Hindi_marianmt']]
bleu_marianmt=corpus_bleu(references,cand_marianmt)
print(f"BLEU Score MarianMT: {bleu_marianmt}")

KeyError: 'Hindi_marianmt'